# SERVERLOGANAMOLY
## Server Log Anomaly Detection with Isolation Forest

### Goal
Parse Apache access logs, aggregate by IP/minute, engineer behavioral features, inject controlled attacks, train Isolation Forest, and evaluate contamination values from 0.01 to 0.05.

In [ ]:
from pathlib import Path
import pandas as pd
from src.log_anomaly import parse_log, engineer_features, train_model, FEATURES, make_report
ROOT=Path.cwd()
if not (ROOT/'data/raw/access.log').exists(): ROOT=ROOT.parent
raw=parse_log(ROOT/'data/raw/access.log')
raw.head()

## 1. Parse logs

In [ ]:
print('Parsed lines:', len(raw))
print('Unique IPs:', raw.ip.nunique())
print(raw.dtypes)

## 2. Engineer IP + 1-minute features

In [ ]:
feat=engineer_features(raw)
feat.head()

In [ ]:
feat[FEATURES].describe().T

## 3. Isolation Forest + contamination comparison

In [ ]:
rows=[]
attack_ips={'198.51.100.10','198.51.100.11','198.51.100.12'}
for c in [0.01,0.02,0.03,0.04,0.05]:
    _,_,res=train_model(feat,c)
    injected=res.ip.isin(attack_ips)
    tp=int((res.anomaly & injected).sum())
    attacks=int(injected.sum())
    fp=int((res.anomaly & ~injected).sum())
    normal=int((~injected).sum())
    rows.append({'contamination':c,'attack_windows':attacks,'attack_windows_flagged':tp,'detection_rate':tp/attacks,'false_positives':fp,'false_positive_rate':fp/normal})
eval_df=pd.DataFrame(rows)
eval_df

### Contamination choice
The default is 0.03 for this controlled fixture. It detects all injected attack windows while keeping the false-positive rate near the configured anomaly budget. On real traffic, tune this value using representative validation data and operational alert capacity.

## 4. Train final model and create Threat Intelligence Report

In [ ]:
model,scaler,result=train_model(feat,0.03)
flagged=result[result.anomaly]
print('Flagged windows:', len(flagged))
flagged.sort_values('anomaly_score').head(20)

In [ ]:
out=ROOT/'reports/notebook_threat_report.md'
make_report(result,raw,out)
print(out)

## 5. Inference workflow
Use the packaged CLI:

`python infer.py path/to/new_access.log --contamination 0.03 --report reports/inference_report.md`